In [ ]:
import pandas as pd
import glob

# Tìm file đã làm sạch
file_paths = glob.glob(r'..\processed\temp_cleaned\cleaned_yellow_tripdata_*.parquet')

file_paths.sort() 


list_of_dataframes = []

for path in file_paths:
    df_month = pd.read_parquet(path)
    list_of_dataframes.append(df_month)

# Gộp tất cả các DataFrame thành một DataFrame duy nhất
df_all_data = pd.concat(list_of_dataframes, ignore_index=True)

df_all_data.to_parquet(r'..\processed\temp_cleaned\all_cleaned_yellow_tripdata_2022.parquet', index=False)

In [2]:
df_all_data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,trip_duration_minutes,trip_speed_mph
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,...,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0,17.816667,12.797007
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,...,0.5,0.5,4.00,0.0,0.3,13.30,0.0,0.0,8.400000,15.000000
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,...,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0,8.966667,6.490706
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,...,0.5,0.5,0.00,0.0,0.3,11.80,2.5,0.0,10.033333,6.518272
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,...,0.5,0.5,3.00,0.0,0.3,30.30,2.5,0.0,37.533333,6.873890


In [1]:
import pandas as pd

# Các cột cần dùng để tính KPI
columns_needed = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
]

file_path = r'..\processed\temp_cleaned\all_cleaned_yellow_tripdata_2022.parquet'
df_kpi_dow= pd.read_parquet(file_path, columns=columns_needed)

df_kpi_dow['Trip_Duration'] = (df_kpi_dow['tpep_dropoff_datetime'] - df_kpi_dow['tpep_pickup_datetime']).dt.total_seconds() / 60

df_kpi_dow['Day_of_Week'] = df_kpi_dow['tpep_pickup_datetime'].dt.day_name()

# Tính toán P95 Trip Duration theo thứ trong tuần
kpi_dow = df_kpi_dow.groupby('Day_of_Week').agg(
    p95_trip_duration=('Trip_Duration', lambda x: x.quantile(0.95))
)

# Sắp xếp thứ trong tuần
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
kpi_dow = kpi_dow.reindex(weekday_order)

# Lưu kết quả
output = r'..\processed\kpi_dow_p95_trip_duration.csv'
kpi_dow.to_csv(output)